# Azure Blob Storage Setup

This notebook provides a comprehensive guide to setting up Azure Blob Storage for ML model artifacts using the Azure Python SDK.

## Prerequisites

1. Install required packages:
```python
!pip install azure-storage-blob azure-identity azure-mgmt-resource azure-mgmt-storage python-dotenv
```

2. Configure environment variables in a `.env` file:
```bash
AZURE_SUBSCRIPTION_ID=your_subscription_id
AZURE_TENANT_ID=your_tenant_id
AZURE_CLIENT_ID=your_client_id
AZURE_CLIENT_SECRET=your_client_secret
AZURE_STORAGE_ACCOUNT_NAME=your_storage_account_name
AZURE_STORAGE_CONTAINER_NAME=models
```

Or use connection string:
```bash
AZURE_STORAGE_CONNECTION_STRING=your_connection_string
```

In [ ]:
# Import required libraries
import os
import uuid
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Get configuration from environment
SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID", "")
TENANT_ID = os.getenv("AZURE_TENANT_ID", "")
CLIENT_ID = os.getenv("AZURE_CLIENT_ID", "")
CLIENT_SECRET = os.getenv("AZURE_CLIENT_SECRET", "")
STORAGE_ACCOUNT_NAME = os.getenv("AZURE_STORAGE_ACCOUNT_NAME", "")
RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP", "rg-mlops")
LOCATION = os.getenv("AZURE_LOCATION", "eastus")
CONTAINER_NAME = os.getenv("AZURE_STORAGE_CONTAINER_NAME", "models")
CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING", "")

print(f"📦 Storage Account: {STORAGE_ACCOUNT_NAME or 'Not set'}")
print(f"📍 Location: {LOCATION}")
print(f"📁 Container Name: {CONTAINER_NAME}")
print(f"🔗 Connection String: {'Configured' if CONNECTION_STRING else 'Not set'}")

## Step 1: Authenticate using Service Principal (Recommended for automation)

In [ ]:
from azure.identity import ClientSecretCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.storage import StorageManagementClient
from azure.storage.blob import BlobServiceClient

# Authenticate using Service Principal
if SUBSCRIPTION_ID and TENANT_ID and CLIENT_ID and CLIENT_SECRET:
    credential = ClientSecretCredential(
        tenant_id=TENANT_ID,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )
    
    # Initialize resource and storage management clients
    resource_client = ResourceManagementClient(credential, SUBSCRIPTION_ID)
    storage_client = StorageManagementClient(credential, SUBSCRIPTION_ID)
    
    print("✅ Authenticated using Service Principal")
else:
    print("⚠️ Service Principal credentials not set. Using DefaultAzureCredential...")
    from azure.identity import DefaultAzureCredential
    credential = DefaultAzureCredential()
    
    # Initialize management clients
    resource_client = ResourceManagementClient(credential, os.getenv("AZURE_SUBSCRIPTION_ID", ""))
    storage_client = StorageManagementClient(credential, os.getenv("AZURE_SUBSCRIPTION_ID", ""))

## Step 2: Create Resource Group

In [ ]:
def create_resource_group(resource_client, resource_group_name, location):
    """Create a resource group if it doesn't exist."""
    try:
        rg = resource_client.resource_groups.get(resource_group_name)
        print(f"ℹ️ Resource group '{resource_group_name}' already exists")
        return rg
    except:
        rg = resource_client.resource_groups.create_or_update(
            resource_group_name,
            {"location": location}
        )
        print(f"✅ Resource group '{resource_group_name}' created in {location}")
        return rg

# Create resource group
resource_group = create_resource_group(resource_client, RESOURCE_GROUP, LOCATION)

## Step 3: Create Storage Account

In [ ]:
def create_storage_account(storage_client, resource_group_name, account_name, location):
    """Create a storage account if it doesn't exist."""
    try:
        # Check if account exists
        storage_account = storage_client.storage_accounts.get_properties(
            resource_group_name,
            account_name
        )
        print(f"ℹ️ Storage account '{account_name}' already exists")
        return storage_account
    except:
        # Create storage account
        poller = storage_client.storage_accounts.begin_create(
            resource_group_name,
            account_name,
            {
                "location": location,
                "sku": {"name": "Standard_LRS"},
                "kind": "StorageV2"
            }
        )
        storage_account = poller.result()
        print(f"✅ Storage account '{account_name}' created")
        return storage_account

# Generate unique storage account name if not provided
if not STORAGE_ACCOUNT_NAME:
    STORAGE_ACCOUNT_NAME = f"mlops{uuid.uuid4().hex[:8]}"

# Create storage account
storage_account = create_storage_account(storage_client, RESOURCE_GROUP, STORAGE_ACCOUNT_NAME, LOCATION)

## Step 4: Get Connection String

In [ ]:
def get_connection_string(storage_client, resource_group_name, account_name):
    """Get connection string for the storage account."""
    keys = storage_client.storage_accounts.list_keys(resource_group_name, account_name)
    conn_string = (
        f"DefaultEndpointsProtocol=https;"
        f"AccountName={account_name};"
        f"AccountKey={keys.keys[0].value};"
        f"EndpointSuffix=core.windows.net"
    )
    return conn_string

# Get connection string
CONNECTION_STRING = get_connection_string(storage_client, RESOURCE_GROUP, STORAGE_ACCOUNT_NAME)
print(f"✅ Got connection string for '{STORAGE_ACCOUNT_NAME}'")

## Step 5: Create Blob Container

In [ ]:
# Connect using connection string
blob_service_client = BlobServiceClient.from_connection_string(CONNECTION_STRING)

def create_container(blob_service_client, container_name):
    """Create a blob container if it doesn't exist."""
    try:
        container_client = blob_service_client.get_container_client(container_name)
        if container_client.exists():
            print(f"ℹ️ Container '{container_name}' already exists")
            return container_client
        
        container_client = blob_service_client.create_container(container_name)
        print(f"✅ Container '{container_name}' created successfully")
        return container_client
    except Exception as e:
        print(f"❌ Error creating container: {e}")
        raise

# Create the container
container_client = create_container(blob_service_client, CONTAINER_NAME)

## Step 6: Upload File to Blob Storage

In [ ]:
def upload_blob_file(blob_service_client, container_name, local_file_path, blob_name=None):
    """Upload a local file to blob storage."""
    if blob_name is None:
        blob_name = os.path.basename(local_file_path)
    
    try:
        container_client = blob_service_client.get_container_client(container_name)
        blob_client = container_client.get_blob_client(blob_name)
        
        with open(local_file_path, "rb") as data:
            blob_client.upload_blob(data, overwrite=True)
        
        print(f"✅ Uploaded '{local_file_path}' as '{blob_name}'")
        return blob_client
    except FileNotFoundError:
        print(f"❌ File not found: {local_file_path}")
    except Exception as e:
        print(f"❌ Upload failed: {e}")
        raise

# Example: Upload a file (uncomment and modify for your use case)
# upload_blob_file(blob_service_client, CONTAINER_NAME, "./my_model.pt", "models/my_model.pt")

## Step 7: List Blobs in Container

In [ ]:
def list_blobs(blob_service_client, container_name, prefix=None):
    """List all blobs in a container."""
    try:
        container_client = blob_service_client.get_container_client(container_name)
        blobs = container_client.list_blobs(name_starts_with=prefix)
        
        blob_list = []
        for blob in blobs:
            print(f"  - {blob.name} ({blob.size} bytes)")
            blob_list.append(blob.name)
        
        return blob_list
    except Exception as e:
        print(f"❌ Error listing blobs: {e}")
        raise

# List all blobs
print(f"📋 Blobs in '{CONTAINER_NAME}':")
blob_list = list_blobs(blob_service_client, CONTAINER_NAME)

## Step 8: Download Blob from Storage

In [ ]:
def download_blob_file(blob_service_client, container_name, blob_name, download_path):
    """Download a blob to local file."""
    try:
        container_client = blob_service_client.get_container_client(container_name)
        blob_client = container_client.get_blob_client(blob_name)
        
        # Create directory if it doesn't exist
        local_dir = os.path.dirname(download_path)
        if local_dir and not os.path.exists(local_dir):
            os.makedirs(local_dir)
        
        with open(download_path, "wb") as download_file:
            download_file.write(blob_client.download_blob().readall())
        
        print(f"✅ Downloaded '{blob_name}' to '{download_path}'")
    except Exception as e:
        print(f"❌ Download failed: {e}")
        raise

# Example: Download a blob (uncomment and modify for your use case)
# download_blob_file(blob_service_client, CONTAINER_NAME, "models/my_model.pt", "./downloads/my_model.pt")

## Summary

In [ ]:
# Print summary
print("=" * 50)
print("📊 Azure Blob Storage Setup Complete!")
print("=" * 50)
print(f"Storage Account: {STORAGE_ACCOUNT_NAME}")
print(f"Container: {CONTAINER_NAME}")
print(f"Location: {LOCATION}")
print("\n📝 Add these to your .env file:")
print(f'AZURE_STORAGE_ACCOUNT_NAME={STORAGE_ACCOUNT_NAME}')
print(f'AZURE_STORAGE_CONTAINER_NAME={CONTAINER_NAME}')
print("\n🔑 Connection string (for reference):")
print(f"{CONNECTION_STRING[:50]}...")